# r3con — provenance

**TL;DR** — r3con records where an answer came from:

- the answer cites the documents it used (`result.provenance`);
- each relevance summary cites the other documents' summaries it used (`result.relevance_provenance`);
- each document's parse cites the summaries it used (`result.parsing_provenance`);
- each parsed record names its document (`"document"` in `result.structured_context`).

They cite documents by id, and `result.document_ids[i]` is the id of `documents[i]`.
The memos and the question are the [quickstart](quickstart.ipynb)'s.

In [1]:
import os
from pathlib import Path

from r3con import r3con

# The key comes from the provider's own variable, or a .env beside this notebook.
assert os.environ.get("OPENAI_API_KEY") or os.path.exists(".env"), (
    "Set OPENAI_API_KEY first."
)

MODEL = "openai/gpt-6-luna"  # any litellm model string

QUESTION = (
    "Which maintenance contractor's sites logged the most equipment incidents in Q3 "
    "in total, and how many? Give the contractor's name, not its code."
)

In [2]:
docs = r3con.read_documents("memos")
result = r3con.run(question=QUESTION, documents=docs, model=MODEL)
print(result)

Halloran Services Ltd's sites logged the most equipment incidents in Q3, with 11 in total.


In [3]:
# read_documents reads a folder in sorted path order, so names[i] is docs[i]'s file.
names = [p.name for p in sorted(Path("memos").glob("*.txt"))]
name_of = dict(zip(result.document_ids, names, strict=True))

print("ids          :", name_of)
print("answer cites :", [name_of[id_] for id_ in result.provenance])

ids          : {'0740f9': '01_northgate.txt', 'ed6bc2': '02_eastfield.txt', 'e2de46': '03_riverside.txt', 'dd5704': '04_contractor_registry.txt', '19660e': '05_cafeteria.txt'}
answer cites : ['01_northgate.txt', '03_riverside.txt', '02_eastfield.txt', '04_contractor_registry.txt']


In [4]:
for id_ in result.provenance:
    i = result.document_ids.index(id_)
    summary = result.relevant_context[i].strip().partition("\n")[0]
    # a part of a document read in parts is cited as <id>.<k>
    drew_on = [name_of[ref.split(".")[0]] for ref in result.relevance_provenance[i]]
    print(f"{name_of[id_]}\n  summary : {summary[:100]}\n  drew on : {drew_on}\n")

01_northgate.txt
  summary : A Northgate Q3 operations memo reports 5 equipment incidents and identifies maintenance contractor c
  drew on : ['04_contractor_registry.txt']

03_riverside.txt
  summary : A Q3 operations-review memo for the Riverside site. It logs 6 equipment incidents and identifies the
  drew on : ['04_contractor_registry.txt']

02_eastfield.txt
  summary : A Q3 operations-review memo for the Eastfield site. It reports 9 equipment incidents and identifies 
  drew on : ['04_contractor_registry.txt']

04_contractor_registry.txt
  summary : An approved contractor registry mapping site-paperwork codes to contractor names. It identifies CT-1
  drew on : ['01_northgate.txt', '02_eastfield.txt', '03_riverside.txt']



In [5]:
for field, records in result.structured_context.items():
    for record in records:
        if record.get("document") in result.provenance:
            print(f"{name_of[record['document']]}  {field}: {record}")

01_northgate.txt  equipment_incident_reports: {'document': '0740f9', 'site': 'Northgate', 'period': 'Q3', 'incident_type': 'equipment incidents', 'incident_count': 5, 'contractor_code': 'CT-118', 'contractor_name': None}
02_eastfield.txt  equipment_incident_reports: {'document': 'ed6bc2', 'site': 'Eastfield', 'period': 'Q3', 'incident_type': 'equipment incidents', 'incident_count': 9, 'contractor_code': 'CT-204', 'contractor_name': None}
03_riverside.txt  equipment_incident_reports: {'document': 'e2de46', 'site': 'Riverside', 'period': 'Q3', 'incident_type': 'equipment incidents', 'incident_count': 6, 'contractor_code': 'CT-118', 'contractor_name': None}
04_contractor_registry.txt  contractor_mappings: {'document': 'dd5704', 'contractor_code': 'CT-118', 'contractor_name': 'Halloran Services Ltd'}
04_contractor_registry.txt  contractor_mappings: {'document': 'dd5704', 'contractor_code': 'CT-204', 'contractor_name': 'Merrow Facilities Group'}
04_contractor_registry.txt  contractor_mappin

In the run folder, `result.run_dir`, the same ids are in `ids.json` (id → index in `documents`), `relevance/result.json` (each round's summaries and what they cite), `structuring/parsing/result.json` (the records and what each document's parse cites) and `reasoning/result.json` (the answer and what it cites).